# Money vs Happiness — Easterlin Paradox

Analiza odnosa bogatstva i sreće na podacima **World Happiness Report 2023**.

**Cilj:** predvidjeti *Happiness score* iz socio-ekonomskih faktora i provjeriti da li veza GDP ↔ sreća "spljoštava" kod najbogatijih zemalja.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from data_loader import FEATURE_COLUMNS, TARGET_COLUMN, get_full_dataframe, load_and_prepare
from eda import (
    analyze_predictor_redundancy,
    basic_statistics,
    plot_correlation_heatmap,
    plot_gdp_vs_happiness,
    plot_happiness_distribution,
    run_eda,
)
from model import (
    plot_feature_importance,
    plot_model_comparison,
    run_clustering,
    run_full_pipeline,
    run_split_sensitivity,
    train_and_evaluate,
)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
print("Project root:", ROOT)

## 1. Učitavanje i priprema podataka

In [ ]:
df = get_full_dataframe()
X, y, meta = load_and_prepare()

print(f"Broj zemalja: {len(df)}")
print(f"Karakteristike: {FEATURE_COLUMNS}")
print(f"Ciljna varijabla: {TARGET_COLUMN}")
df.head()

## 2. Eksploratorna analiza (EDA)

In [ ]:
basic_statistics(df)

In [ ]:
plot_happiness_distribution(df)
plt.show()

In [ ]:
plot_correlation_heatmap(df)
plt.show()

### Međusobne korelacije 6 faktora (da li izbaciti kolonu?)

Pearson matrica **samo među prediktorima** pokazuje da li dva faktora nose istu informaciju. Prag: **|r| ≥ 0.8** je kandidat za redundantnost; **VIF > 10** bi dodatno opravdao izbacivanje. U ovom skupu zadržavamo svih 6 kolona.

In [ ]:
redundancy = analyze_predictor_redundancy(df)
display(redundancy["pairs"])
display(redundancy["vif"])
print("Izbacene kolone:", redundancy["dropped_columns"] or "nijedna")

### Ključni grafikon: GDP vs Happiness (Easterlin Paradox)

Ako kvadratna krivulja pokazuje **negativnu zakrivljenost** (spljoštavanje kod visokog GDP-a), to ide u prilog Easterlin paradoxa — bogatstvo nakon određene tačke daje sve manji prirast sreće.

In [ ]:
plot_gdp_vs_happiness(df)
plt.show()

## 3. Modelovanje — Linear Regression, Random Forest, XGBoost

### Robustnost splita: 90/10, 80/20, 70/30, 60/40

Glavni rezultat ostaje **80/20**. Ovdje treniram iste tri modele na drugim omjerima (i na više 80/20 shuffle-ova) da vidim da li se model poboljša ili pokvari.

In [ ]:
split_result = run_split_sensitivity(X, y)
display(split_result["ratios"][["Split", "n_train", "n_test", "Model", "Test_R2", "Test_MAE"]])
display(split_result["seeds"][["random_state", "Model", "Test_R2", "Test_MAE"]])

In [ ]:
metrics_df, cv_df, fitted, (X_train, X_test, y_train, y_test) = train_and_evaluate(X, y)
metrics_df

In [ ]:
plot_model_comparison(metrics_df)
plt.show()

## 4. Interpretacija — Feature Importance

In [ ]:
plot_feature_importance(fitted, list(X.columns))
plt.show()

## 5. K-Means klasterovanje zemalja

In [ ]:
clustered = run_clustering(X, y, meta, n_clusters=3)
clustered.groupby("Cluster label")[[TARGET_COLUMN, "Logged GDP per capita"]].mean()

In [ ]:
print("Primjeri zemalja po klasteru:")
for label in ["Lower happiness", "Mid happiness", "Higher happiness"]:
    countries = clustered.loc[clustered["Cluster label"] == label, "Country name"].head(5).tolist()
    print(f"  {label}: {', '.join(countries)}")

## 6. Zaključak

1. **Social support**, **GDP** i **Healthy life expectancy** su najjače povezani sa Happiness score-om.
2. Na ovom skupu **Linear Regression** često daje najbolji R²; XGBoost ne mora biti najbolji na malom cross-section uzorku.
3. Scatter GDP–sreća pokazuje jaku pozitivnu vezu; Easterlin paradox je više o *vremenskom* trendu unutar zemalja nego o tome da bogate zemlje nisu sretne.
4. Feature importance i K-Means potvrđuju da sreća nije samo GDP — socijalni profili zemlje su ključni.

Grafikoni su sačuvani u `results/plots/`.